<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/leaderboard_focus/00_ENVIRONMENT_AND_ARTIFACTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M1 · 00 — Environment and artifact recovery (CPU)

**Purpose.** Pin the code revision, inventory every frozen artifact Milestone 1 needs on Google Drive (existence, count, sha256, experiment ids), stage only the small files locally, and record the result. No training, no inference, no checkpoint download.

| | |
|---|---|
| Runtime | CPU (standard RAM is enough) |
| Prerequisites | `MyDrive/GEMS_2` with the P6 outputs (`outputs/p6_fast_track/{stack,boost,recovery}`), the B0 / B1 experiments, folds, template |
| Inputs | the artifact list in `configs/leaderboard_focus/m1.yaml` → `artifacts` |
| Outputs | `outputs/leaderboard_focus/m1_champion_recovery/{environment/environment.json, artifact_inventory.csv, artifact_inventory.json, staging_manifest.json, manifests/artifacts.json}` (local, then Drive) |
| Next | `01_CHAMPION_RECOVERY.ipynb` once the inventory status is **COMPLETE** |

## 1 · Configuration

In [1]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                    # pin the pushed commit SHA for a reproducible run
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'reuse_cached': True,             # reuse staged files while the inventory fingerprint is unchanged
    'strict_provenance': True,        # a changed B0 / B1 experiment id blocks M1
    'overwrite_existing': False,
    'device': 'cpu',
}
PRIVATE_REPO = False
M1_CONFIG = {}                       # overrides of configs/leaderboard_focus/m1.yaml (normally empty)
EXPECT = {'code_rev': 'm1.1-r1', 'commit': None}

## 2 · Drive, strict clone, version lock

A requested ref that cannot be resolved stops here: the notebook never falls back to another branch.

In [2]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'm1_colab_active', 'origin/' + ref)
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)       # a SHA / tag must resolve, else this raises
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: restart the runtime')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
import gems_bench.leaderboard_focus as LF
assert LF.M1_CODE_REV == EXPECT['code_rev'], f'code rev {LF.M1_CODE_REV} != {EXPECT["code_rev"]}'
assert not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']), f'HEAD {HEAD} != {EXPECT["commit"]}'
print('Requested:', ref, '| resolved SHA:', HEAD, '| M1 code rev:', LF.M1_CODE_REV)

Mounted at /content/drive
Requested: main | resolved SHA: bff536539ab326b729f058c87d642aefea1a2213 | M1 code rev: m1.1-r1


## 3 · Inventory, selective staging, verification

Statuses: `OK`, `MISSING`, `MISSING_REQUIRED_FILES`, `COUNT_MISMATCH`, `AMBIGUOUS`, `ID_CHANGED` (a B0 / B1 experiment id differs from the one seen in the last recorded run). A required artifact that is not OK blocks M1.

B0 / B1 prediction folders are checked **by name**: `oof.tif`, `disc0.tif`, `disc1.tif`, `disc2.tif` must exist in exactly one experiment directory with the recorded id. Other GeoTIFFs there (the per-fold `oof_fold<k>.tif`) are accepted, hashed into the fingerprint and listed in `extra_files`; they are never an error and never deleted or renamed. Matches in several experiment directories are `AMBIGUOUS` (never combined).

Only files marked `staged` are copied (W2 maps, grids, recovery tables, selection files); CNN prediction rasters are staged later by the pipeline, checkpoints never.

In [3]:
import json
from IPython.display import display
from gems_bench.leaderboard_focus.artifacts import run_environment
ENV = run_environment(CONFIG, HEAD, M1_CONFIG)
S = ENV['summary']
print('ARTIFACT STATUS:', S['status'], '| fingerprint', S['fingerprint'], '| counts', S['counts'])
for b in S['blocking']:
    print('  BLOCKING:', b)
print('Optional missing:', S['optional_missing'])
display(ENV['inventory'][['id', 'required', 'status', 'n_files', 'experiment_ids', 'required_files', 'extra_files',
                          'note']])
print('Staged files:', ENV['staging']['n_files'], '| mismatches:', ENV['staging']['mismatch'], '|', ENV['staging']['note'])

[sync in] 17 copied (263.2 MiB), 0 unchanged, 2s: ['outputs/m9_geofault_production/baseline/baseline_selection.json', 'outputs/p6_fast_track/boost/benchmark/grid.csv', 'outputs/p6_fast_track/boost/benchmark/manifest.json', 'outputs/p6_fast_track/recovery/export_parity/decision.json', 'outputs/p6_fast_track/recovery/export_parity/policy_comparison.csv', 'outputs/p6_fast_track/recovery/export_parity/train_inference_parity.csv', 'outputs/p6_fast_track/recovery/export_parity/unit_overlap.csv', 'outputs/p6_fast_track/stack/candidate.json', 'outputs/p6_fast_track/stack/features/rows.parquet', 'outputs/p6_fast_track/stack/grid.csv', 'outputs/p6_fast_track/stack/maps/disc_official_0.npy', 'outputs/p6_fast_track/stack/maps/disc_official_1.npy', 'outputs/p6_fast_track/stack/maps/disc_official_2.npy', 'outputs/p6_fast_track/stack/maps/full_score.tif', 'outputs/p6_fast_track/stack/maps/geo_0.npy', 'outputs/p6_fast_track/stack/maps/geo_1.npy', 'outputs/p6_fast_track/stack/maps/geo_2.npy']
[sync out

,id,required,status,n_files,experiment_ids,required_files,extra_files,note
0,w2_candidate,True,OK,1,,[],[],
1,w2_rows,True,OK,1,,[],[],
2,w2_stack_grid,True,OK,1,,[],[],
3,w2_maps,True,OK,6,,[],[],
4,w2_full_score_hist,False,OK,1,,[],[],
5,boost_a_grid,True,OK,1,,[],[],
6,boost_a_manifest,True,OK,1,,[],[],
7,recovery_decision,False,OK,1,,[],[],
8,recovery_parity,False,OK,1,,[],[],
9,recovery_policies,False,OK,1,,[],[],


Staged files: 17 | mismatches: [] | staged


## 4 · Verification checklist

1. `ARTIFACT STATUS: COMPLETE` and no `BLOCKING` line.
2. `experiment_ids` of `b0_manifest` / `b0_maps` are `b0_reference__d54e938a`, of `b1_manifest` / `b1_maps` are `b1_bg_weight__7755635b`.
3. `b0_maps` / `b1_maps`: `required_files` lists all four names; `extra_files` lists any additional GeoTIFFs (expected: `oof_fold0.tif`, `oof_fold1.tif`, `oof_fold2.tif`).
4. `staging_manifest.json` → `mismatch` is empty.
5. The files exist on Drive under `MyDrive/GEMS_2/outputs/leaderboard_focus/m1_champion_recovery/`.

## 5 · Troubleshooting

* `MISSING w2_maps / boost_a_grid`: the P6 outputs are not on this Drive. Restore `outputs/p6_fast_track/` (M1 never recomputes them).
* `MISSING_REQUIRED_FILES b0_maps / b1_maps`: one of `oof.tif`, `disc0.tif`, `disc1.tif`, `disc2.tif` is absent from the experiment's `predictions/` folder; the note lists what was found. Restore it from the original run (M1 never re-infers).
* `AMBIGUOUS b0_maps / b1_maps`: several `b0_reference__*` / `b1_bg_weight__*` folders match. M1 never combines or picks one. Identify the recorded run from `experiment_ids`; resolving the duplicate is a manual decision (M1 deletes, moves and renames nothing).
* `ID_CHANGED`: a newer B0 / B1 run exists. Confirm it is the intended model, then set `expected_id` in `M1_CONFIG['artifacts']` explicitly (never by default).
* `template MISSING`: run the pipeline `raw_data` step once (any earlier notebook) to fetch `example_submission.tif`.
* Re-running is safe: files are written atomically; staging is skipped while the fingerprint is unchanged. A previous BLOCKED `artifact_inventory.json` is simply overwritten.

## 6 · Next

Open `01_CHAMPION_RECOVERY.ipynb` (CPU, High-RAM recommended).